### Functions

In [1]:
def load_session_data(subject, date):
    """Load all data for a given subject and date"""
    import sys
    sys.path.append(r'D:\Neural-Pipeline\source')
    from analysis_utils.NeuralDataLoader import NeuralDataLoader, Dots3DMPConfig
  
    
    # Load session
    loader = NeuralDataLoader()
    loader.load_session(subject, date)
    config = Dots3DMPConfig(subject)

    # spike data (unit, trial, time)
    stimOn_spikes = loader.get_spike_data(alignment='stimOn', good_units_only=True, good_trials_only=True)
    saccOnset_spikes = loader.get_spike_data(alignment='saccOnset', good_units_only=True, good_trials_only=True)
    postTargHold_spikes = loader.get_spike_data(alignment='postTargHold', good_units_only=True, good_trials_only=True)
    tuning_spikes = loader.get_tuning_data(good_units_only=True, good_trials_only=True)

    # behavioral data
    behavior_dots3DMP = loader.get_behavioral_data(task='dots3DMP', good_trials_only=True)
    behavior_tuning = loader.get_behavioral_data(task='tuning', good_trials_only=True)
    behavior_converted = config.convert_behavioral_data(behavior_dots3DMP, task='dots3DMP')
    behavior_tuning_converted = config.convert_behavioral_data(behavior_tuning, task='tuning')

    # Unit Info
    unit_info = loader.get_unit_info(good_units_only=True)
    MST_units = loader.get_units_by_area(unit_info, area_name='MST')
    VPS_units = loader.get_units_by_area(unit_info, area_name='VPS')
    dual_units = loader.get_units_by_area(unit_info, area_name='dual')

    # Time Info
    time_info = config.get_time_Info('dots3DMP')
    time_info_tuning = config.get_time_Info('tuning')
    time_axes_dots3DMP = config.get_time_axes('dots3DMP')

    # Prepare data
    spikes_data = {
        'stimOn': stimOn_spikes,
        'saccOnset': saccOnset_spikes,
        'postTargHold': postTargHold_spikes
    }
    
    units_data = {
        'MST': MST_units,
        'VPS': VPS_units,
        'dual': dual_units
    }
    
    return {
        'loader': loader,
        'config': config,
        'spikes_data': spikes_data,
        'behavior_converted': behavior_converted,
        'behavior_tuning_converted': behavior_tuning_converted,
        'unit_info': unit_info,
        'units_data': units_data,
        'time_axes_dots3DMP': time_axes_dots3DMP,
        'time_info': time_info,
        'time_info_tuning': time_info_tuning
    }

In [2]:
def run_sliding_window_dPCA(subject, date, data_dict):
    """Run sliding window dPCA analysis"""
    from analysis_population.PopulationdPCA import SlidingWindowdPCA
                

    areas = ['dual', 'MST', 'VPS']
    
    # Run decoding analysis
    for area in areas:
        print(f"\n=== Processing area: {area} ===")
        
        valid_units = data_dict['units_data'][area]
        
            
        # Create decoder instance
        d_PCA = SlidingWindowdPCA(subject, date)
        
        for mod in [1, 2, 3]:
            for coh in [1, 2]:
                if mod == 1 and coh == 2:
                    continue  

                results = d_PCA.run_dpca_analysis(
                    spikes_data=data_dict['spikes_data'],
                    behavior_data=data_dict['behavior_converted'],
                    time_axes=data_dict['time_axes_dots3DMP'],
                    area=area,
                    train_mod=mod, 
                    train_coh=coh,    
                    valid_units=valid_units,
                    save_results=True
                )

    print(f"Completed sliding window decoding for {subject} {date}")

In [3]:

def process_all_dates(subject, dates):

    
    for date in dates:
        print(f"\n{'#'*80}")
        print(f"PROCESSING DATE: {date}")
        print(f"{'#'*80}")
        try:
            # Step 1: Load data
            print("Loading session data...")
            data_dict = load_session_data(subject, date)
            
            # Step 2: run dPCA 
            run_sliding_window_dPCA(subject, date, data_dict)

            

        except Exception as e:
            print(f"Error processing date {date}: {e}")
            continue
        

### Main code, run here

In [4]:
subject = "zarya"
dates = ["20250602", "20250702", "20250710", "20250523", "20250501", "20250417"]


# Run both hyperparameter search and sliding window decoding
process_all_dates(subject, dates)


################################################################################
PROCESSING DATE: 20250602
################################################################################
Loading session data...
Loaded dots3DMP data: zarya20250602dots3DMP_processed.npz
Loaded dots3DMPtuning data: zarya20250602dots3DMPtuning_processed.npz
Loaded dots3DMP configuration for subject: zarya

=== Processing area: dual ===

RUNNING FULL SPIKE TRAIN dPCA ANALYSIS
Subject: zarya, Date: 20250602
Area: dual, Condition: mod1_coh1_del0

----------------------------------------
Processing alignment: stimOn
----------------------------------------
Organizing data by heading and choice conditions...
Complete conditions: 8/14
Using parametric linear model to fill missing conditions...
Fitting linear models to fill missing data...
Linear model fitting completed!
dPCA data shape: (125, 51, 7, 2)
Valid conditions: 8/14
Full spike train dPCA results saved to: D:\Neural-Pipeline\results\analysis_population